In [7]:
import os
from dotenv import load_dotenv
from huggingface_hub import notebook_login, whoami
from langchain_community.agent_toolkits.load_tools import load_tools

from smolagents import CodeAgent, InferenceClientModel, tool, Tool, load_tool

load_dotenv();
notebook_login()

print(whoami()["name"])

stepit


We can create tools using the `@tool` decorator:

In [2]:
@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.

    Args:
        query: A search term for finding catering services.
    """

    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    # Find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)

    return best_service


model = InferenceClientModel(model_id="meta-llama/Llama-3.3-70B-Instruct")

agent = CodeAgent(tools=[catering_service_tool], model=model)

result = agent.run(
    "Can you give me the name of the highest-rated catering service in Gotham City?"
)

print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Can you give me the name of the highest-rated catering service in Gotham City?                                  │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.3-70B-Instruct ──────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  highest_rated_catering = catering_service_tool(query="highest-rated catering service in Gotham City")            
  print(highest_rated_catering)                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Gotham Catering Co.

Out: None

[Step 1: Duration 1.17 seconds| Input tokens: 2,063 | Output tokens: 74]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Gotham Catering Co.")                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Gotham Catering Co.

[Step 2: Duration 1.69 seconds| Input tokens: 4,293 | Output tokens: 134]

Gotham Catering Co.


Or we can define a tool by inheriting from the `Tool` class:

In [3]:
class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""

    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic Gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }

        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")

party_theme_tool = SuperheroPartyThemeTool()

In [4]:
model = InferenceClientModel(model_id="meta-llama/Llama-3.3-70B-Instruct")

agent = CodeAgent(tools=[party_theme_tool], model=model)

result = agent.run("What would be a good superhero party idea for a 'villain masquerade' theme?")

print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What would be a good superhero party idea for a 'villain masquerade' theme?                                     │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.3-70B-Instruct ──────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  theme = superhero_party_theme_generator(category='villain masquerade')                                           
  print(theme)                                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

Out: None

[Step 1: Duration 1.54 seconds| Input tokens: 2,098 | Output tokens: 74]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.")      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

[Step 2: Duration 1.52 seconds| Input tokens: 4,366 | Output tokens: 176]

Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.


If you have an HF Pro account you can also push your tool to the Hub:

In [5]:
#party_theme_tool.push_to_hub("stepit/catering_service_tool", token=HF_TOKEN)

And we can import tools too:

In [10]:
image_generation_tool = load_tool(
    "m-ric/text-to-image",
    trust_remote_code=True
)

model = InferenceClientModel(model_id="meta-llama/Llama-3.3-70B-Instruct")

agent = CodeAgent(tools=[image_generation_tool, party_theme_tool], model=model)

agent.run("Generate an image of a luxurious superhero-themed party at Wayne Manor with made-up superheros.")

NameError: name 'party_theme_tool' is not defined

Or we can import tools directly from Hugging Face Spaces. The following code is broken until this PR will be merged: https://github.com/huggingface/smolagents/pull/2485

In [8]:
if False:
    image_generation_tool = Tool.from_space(
        "black-forest-labs/FLUX.1-schnell",
        name="image_generator",
        description="Generate an image from a prompt"
    )

    model = InferenceClientModel(model_id="meta-llama/Llama-3.3-70B-Instruct")

    agent = CodeAgent(tools=[image_generation_tool], model=model)

    agent.run(
        "Improve this prompt, then generate an image of it.",
        additional_args={'user_prompt': 'A grand superhero-themed party at Wayne Manor, with Alfred overseeing a luxurious gala'}
    )

LangChain also provide access to tools:

In [9]:
search_tool = Tool.from_langchain(load_tools(["serpapi"])[0])

model = InferenceClientModel(model_id="meta-llama/Llama-3.3-70B-Instruct")

agent = CodeAgent(tools=[search_tool], model=model)

agent.run("Search for luxury entertainment ideas for a superhero-themed event, such as live performances and interactive experiences.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for luxury entertainment ideas for a superhero-themed event, such as live performances and interactive   │
│ experiences.                                                                                                    │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.3-70B-Instruct ──────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_query = "luxury entertainment ideas for superhero-themed events"                                          
  results = search(search_query)                                                                                   
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
['IDEAS is a central index of economics and finance research, including working papers, articles and software 
code.', 'Meeting & Event Space Revenue Management. Turn meetings and events into a strategic profit driver with 
accurate forecasting and smarter, data-backed decisions.', 'Browse through materials and notes relevant to the 
future of our profession and your own professional development through the Public Library or Members Library.', 
'The Individuals with Disabilities Education Act (IDEA) is a law that makes available a free appropriate public 
education to eligible children with disabilities ...', "Ideas is CBC Radio's program of contemporary thought.", 
'Where fan creations turn into real LEGO® sets. Come explore amazing fan builds, join challenges and cheer on 
creators. Have an idea? Submit it and your design ...', "The New IDEAS Study. Sponsored and managed by: American 
College of Radiology®. Directed by: Alzheimer's Association. Advised by: Centers for Medicare and ...", 'An idea is
the result of thought. Although the word is used in common usage, in philosophy, ideas can also be mental 
representational images of some object.', 'As an administrator, I would like to be able to quickly filter and view 
which users have checked ArcGIS Pro offline. Currently, I can scroll and look for users ...']

Out: None

[Step 1: Duration 4.00 seconds| Input tokens: 2,070 | Output tokens: 86]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Luxury entertainment ideas for a superhero-themed event could include hosting a masquerade ball    
  with a superhero twist, organizing a charity gala with a superhero theme, or creating an immersive experience    
  with superhero characters and storylines.")                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Luxury entertainment ideas for a superhero-themed event could include hosting a masquerade ball with 
a superhero twist, organizing a charity gala with a superhero theme, or creating an immersive experience with 
superhero characters and storylines.

[Step 5: Duration 2.42 seconds| Input tokens: 15,900 | Output tokens: 465]

'Luxury entertainment ideas for a superhero-themed event could include hosting a masquerade ball with a superhero twist, organizing a charity gala with a superhero theme, or creating an immersive experience with superhero characters and storylines.'